In [ ]:
%pip install torch transformers spacy openai datasets accelerate bitsandbytes
%python -m spacy download en_core_web_sm
%pip install -q huggingface_hub

In [ ]:
%pip install -q datasets

In [ ]:
%pip install python-dotenv

In [ ]:
# Phase 1: Environment Setup & Dataset Initialization

import spacy
import requests
from transformers import AutoModel, AutoTokenizer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATASET_URL = (
    "https://raw.githubusercontent.com/zackhuiiiii/Privacy-R1/"
    "main/med_pcd_dataset/med_pcd_all.json"
)


def load_spacy_model():
    try:
        return spacy.load("en_core_web_sm")
    except OSError:
        from spacy.cli import download

        download("en_core_web_sm")
        return spacy.load("en_core_web_sm")


def load_med_pcd_dataset():
    response = requests.get(DATASET_URL, timeout=30)
    response.raise_for_status()
    return response.json()


def initialize_environment():
    print("Setting up the environment...")

    nlp = load_spacy_model()
    print("SpaCy model loaded successfully.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    feature_extractor = AutoModel.from_pretrained(MODEL_NAME)

    for param in feature_extractor.parameters():
        param.requires_grad = False

    print("MiniLM loaded with frozen weights.")

    dataset = load_med_pcd_dataset()
    print(f"Loaded {len(dataset)} Med-PCD records.")

    required_fields = {
        "query",
        "pii_units",
        "target_response"
    }

    for idx, record in enumerate(dataset[:1]):
        missing_fields = required_fields - set(record.keys())

        if missing_fields:
            raise ValueError(
                f"Record {idx} is missing fields: {missing_fields}"
            )

    print("Dataset fields validated successfully.")

    return nlp, tokenizer, feature_extractor, dataset


if __name__ == "__main__":
    nlp, tokenizer, feature_extractor, dataset = initialize_environment()

In [ ]:
# Phase 2: Text Preprocessing & Feature Extraction

import torch
import spacy
from transformers import AutoTokenizer, AutoModel


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def initialize_chunker():
    return spacy.load(
        "en_core_web_sm",
        disable=["ner", "tagger", "lemmatizer", "attribute_ruler"]
    )


def chunk_text(text: str, nlp_model) -> list[str]:
    doc = nlp_model(text)
    return [
        sent.text.strip()
        for sent in doc.sents
        if sent.text.strip()
    ]


class MiniLMFeatureExtractor:

    def __init__(self, device=None):

        self.device = device or (
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )

        self.model = AutoModel.from_pretrained(
            MODEL_NAME
        ).to(self.device)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.eval()

        self.embedding_dim = self.model.config.hidden_size


    def get_embeddings(
        self,
        chunks: list[str]
    ) -> torch.Tensor:

        if not chunks:
            return torch.empty(
                (0, self.embedding_dim),
                device=self.device
            )

        inputs = self.tokenizer(
            chunks,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(self.device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = self.model(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        sum_embeddings = torch.sum(
            token_embeddings * mask,
            dim=1
        )

        sum_mask = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return sum_embeddings / sum_mask

In [ ]:
# Phase 3: Privacy-R1 Agent Architecture

import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model: int,
        max_len: int = 5000
    ):
        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )


    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        seq_len = x.size(1)

        if seq_len > self.pe.size(1):
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"maximum supported length {self.pe.size(1)}."
            )

        return x + self.pe[:, :seq_len, :]


class PrivacyR1Agent(nn.Module):

    def __init__(
        self,
        embedding_dim: int = 384,
        n_heads: int = 8,
        n_layers: int = 2,
        hidden_dim: int = 512
    ):
        super().__init__()

        if embedding_dim % n_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by n_heads."
            )

        self.embedding_dim = embedding_dim

        self.pos_encoder = PositionalEncoding(
            d_model=embedding_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.actor_head = nn.Sequential(
            nn.Linear(embedding_dim, 2),
            nn.Softmax(dim=-1)
        )

        self.critic_head = nn.Linear(
            embedding_dim,
            1
        )


    def forward(
        self,
        embeddings: torch.Tensor
    ):

        x = self.pos_encoder(embeddings)

        contextualized_chunks = self.transformer_encoder(x)

        action_probs = self.actor_head(
            contextualized_chunks
        )

        state_values = self.critic_head(
            contextualized_chunks
        )

        return action_probs, state_values

In [ ]:
# Phase 4: Reinforcement Learning Environment & Reward Function

from huggingface_hub import InferenceClient


JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

_judge_client = None


def calculate_privacy_leak(
    pii_entities,
    remote_chunks
):
    if not pii_entities:
        return 0.0

    remote_text = " ".join(remote_chunks).lower()

    leaked_count = sum(
        str(pii).lower() in remote_text
        for pii in pii_entities
    )

    return leaked_count / len(pii_entities)


def get_judge_client(hf_token):
    global _judge_client

    if _judge_client is None:
        _judge_client = InferenceClient(
            model=JUDGE_MODEL,
            token=hf_token,
            timeout=60
        )

    return _judge_client


def evaluate_task_gain(
    query,
    gold_response,
    generated_response,
    hf_token
):
    if not generated_response or not generated_response.strip():
        return 0.0

    client = get_judge_client(hf_token)

    prompt = f"""
You are a strict evaluator of an LLM response.

QUERY:
{query}

REFERENCE ANSWER:
{gold_response}

GENERATED RESPONSE:
{generated_response}

Evaluate the generated response using this scale:

0 = Incorrect, irrelevant, or not useful.
1 = Partially correct or useful.
2 = Correct, relevant, and sufficiently complete.

Return ONLY one number: 0, 1, or 2.
"""

    try:
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an objective response evaluator. "
                        "Return only the requested score."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=5,
            temperature=0.0
        )

        result = (
            response.choices[0]
            .message
            .content
            .strip()
        )

        if result.startswith("2"):
            return 1.0

        if result.startswith("1"):
            return 0.5

        if result.startswith("0"):
            return 0.0

        return 0.0

    except Exception as e:
        print(f"LLM judge failed: {e}")
        return 0.0


def calculate_reward(
    task_gain,
    privacy_leak,
    lambda_penalty=2.0
):
    return (
        float(task_gain)
        - lambda_penalty * (privacy_leak ** 2)
    )

In [ ]:
# Phase 5: Hybrid Routing & Rollout Execution

import torch
from transformers import pipeline
from huggingface_hub import InferenceClient


LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
REMOTE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

_local_pipe = None
_remote_client = None


def generate_hybrid_response(
    chunks,
    actions,
    local_model_fn,
    remote_model_fn
):
    local_chunks = []
    remote_chunks = []

    for chunk, action in zip(chunks, actions):
        if action == 1:
            remote_chunks.append(chunk)
        else:
            local_chunks.append(chunk)

    local_context = " ".join(local_chunks)
    remote_context = " ".join(remote_chunks)

    local_output = (
        local_model_fn(local_context)
        if local_chunks
        else ""
    )

    remote_output = (
        remote_model_fn(remote_context)
        if remote_chunks
        else ""
    )

    final_response = f"{remote_output} {local_output}".strip()

    return final_response, remote_chunks


def make_local_llm():
    global _local_pipe

    if _local_pipe is None:
        _local_pipe = pipeline(
            "text-generation",
            model=LOCAL_MODEL,
            device_map="auto"
        )

        _local_pipe.model.generation_config.max_length = None

    def _call(text):

        if not text.strip():
            return ""

        output = _local_pipe(
            text,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=_local_pipe.tokenizer.eos_token_id
        )

        generated_text = output[0]["generated_text"]

        return generated_text[len(text):].strip()

    return _call


def make_remote_llm(hf_token):
    global _remote_client

    if not hf_token:
        raise ValueError(
            "Hugging Face token is required for remote inference."
        )

    if _remote_client is None:
        _remote_client = InferenceClient(
            model=REMOTE_MODEL,
            token=hf_token,
            timeout=60
        )

    def _call(text):

        if not text.strip():
            return ""

        response = _remote_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": text
                }
            ],
            max_tokens=256,
            temperature=0.0
        )

        return response.choices[0].message.content.strip()

    return _call


def execute_real_rollout(
    query,
    pii_entities,
    gold_response,
    agent,
    chunker,
    extractor,
    hf_token
):
    chunks = chunk_text(
        query,
        chunker
    )

    if not chunks:
        chunks = [query]

    embeddings = (
        extractor
        .get_embeddings(chunks)
        .unsqueeze(0)
    )

    with torch.no_grad():
        action_probs, state_value = agent(
            embeddings
        )

        distribution = torch.distributions.Categorical(
            action_probs
        )

        actions = distribution.sample()

        log_prob = distribution.log_prob(
            actions
        )

    local_llm = make_local_llm()
    remote_llm = make_remote_llm(hf_token)

    final_response, remote_chunks = generate_hybrid_response(
        chunks,
        actions[0].tolist(),
        local_llm,
        remote_llm
    )

    task_gain = evaluate_task_gain(
        query,
        gold_response,
        final_response,
        hf_token
    )

    privacy_leak = calculate_privacy_leak(
        pii_entities,
        remote_chunks
    )

    reward = calculate_reward(
        task_gain,
        privacy_leak
    )

    return (
        embeddings,
        actions,
        log_prob,
        reward,
        state_value,
        privacy_leak,
        task_gain
    )

In [ ]:
# Phase 6: Supervised Warm-up

import torch
import torch.nn as nn
import torch.optim as optim


def generate_heuristic_labels(
    chunks: list[str],
    pii_entities: list[str]
) -> torch.Tensor:

    labels = []

    for chunk in chunks:
        chunk_lower = chunk.lower()

        has_pii = any(
            str(pii).lower() in chunk_lower
            for pii in pii_entities
        )

        labels.append(0.0 if has_pii else 1.0)

    return torch.tensor(
        labels,
        dtype=torch.float32
    )


def train_supervised_warmup(
    agent: nn.Module,
    dataloader: list,
    epochs: int = 1,
    lr: float = 3e-4
):

    if not dataloader:
        raise ValueError(
            "Supervised warm-up dataloader is empty."
        )

    criterion = nn.BCELoss()

    optimizer = optim.Adam(
        agent.parameters(),
        lr=lr
    )

    agent.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for embeddings, labels in dataloader:

            optimizer.zero_grad()

            action_probs, _ = agent(embeddings)

            remote_probs = action_probs[:, :, 1]

            labels = labels.to(
                remote_probs.device
            )

            loss = criterion(
                remote_probs,
                labels
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        average_loss = (
            total_loss / len(dataloader)
        )

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Average BCE Loss: {average_loss:.4f}"
        )

    return agent

In [ ]:
# Phase 7: PPO Reinforcement Learning & Fine-Tuning

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from collections import deque
from torch.distributions import Categorical
from tqdm.auto import tqdm


class PPOTrainer:

    def __init__(
        self,
        agent,
        lr=1e-5,
        clip_param=0.2,
        gamma=0.99,
        gae_lambda=0.95,
        value_coef=0.5,
        entropy_coef=0.01,
        batch_size=8
    ):
        self.agent = agent
        self.optimizer = optim.Adam(
            self.agent.parameters(),
            lr=lr
        )

        self.clip_param = clip_param
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.batch_size = batch_size

        self.memory = []


    def select_action(self, embeddings):

        action_probs, state_values = self.agent(
            embeddings
        )

        distribution = Categorical(
            action_probs
        )

        actions = distribution.sample()
        log_probs = distribution.log_prob(
            actions
        )

        return (
            actions,
            log_probs,
            state_values
        )


    def store_transition(
        self,
        state,
        action,
        log_prob,
        reward,
        value
    ):

        self.memory.append(
            {
                "state": state.detach(),
                "action": action.detach(),
                "log_prob": log_prob.detach(),
                "reward": float(reward),
                "value": value.detach()
            }
        )


    def _calculate_advantages(self):

        rewards = torch.tensor(
            [item["reward"] for item in self.memory],
            dtype=torch.float32
        )

        values = torch.stack(
            [
                item["value"].mean().detach()
                for item in self.memory
            ]
        )

        advantages = torch.zeros_like(
            rewards
        )

        returns = torch.zeros_like(
            rewards
        )

        gae = 0.0
        next_value = 0.0

        for t in reversed(range(len(rewards))):

            if t == len(rewards) - 1:
                delta = (
                    rewards[t]
                    - values[t]
                )
            else:
                delta = (
                    rewards[t]
                    + self.gamma * values[t + 1]
                    - values[t]
                )

            gae = (
                delta
                + self.gamma
                * self.gae_lambda
                * gae
            )

            advantages[t] = gae

        returns = advantages + values

        if advantages.numel() > 1:
            advantages = (
                advantages - advantages.mean()
            ) / (
                advantages.std() + 1e-8
            )

        return advantages, returns


    def update(self):

        if not self.memory:
            return None

        advantages, returns = (
            self._calculate_advantages()
        )

        total_policy_loss = 0.0
        total_value_loss = 0.0
        total_entropy = 0.0

        self.agent.train()
        self.optimizer.zero_grad()

        for idx, transition in enumerate(
            self.memory
        ):

            state = transition["state"]
            action = transition["action"]
            old_log_prob = transition["log_prob"]

            new_probs, new_values = self.agent(
                state
            )

            new_probs = new_probs.clamp(
                min=1e-8,
                max=1.0 - 1e-8
            )

            distribution = Categorical(
                new_probs
            )

            new_log_prob = distribution.log_prob(
                action
            )

            entropy = distribution.entropy().mean()

            ratio = torch.exp(
                new_log_prob - old_log_prob
            )

            advantage = advantages[idx]

            advantage = advantage.expand_as(
                new_log_prob
            )

            surrogate_1 = (
                ratio * advantage
            )

            surrogate_2 = (
                torch.clamp(
                    ratio,
                    1.0 - self.clip_param,
                    1.0 + self.clip_param
                )
                * advantage
            )

            policy_loss = -torch.min(
                surrogate_1,
                surrogate_2
            ).mean()

            value_prediction = (
                new_values.mean()
            )

            target_return = returns[idx].to(
                value_prediction.device
            )

            value_loss = nn.functional.mse_loss(
                value_prediction,
                target_return
            )

            loss = (
                policy_loss
                + self.value_coef * value_loss
                - self.entropy_coef * entropy
            )

            loss = loss / len(self.memory)

            loss.backward()

            total_policy_loss += (
                policy_loss.item()
            )

            total_value_loss += (
                value_loss.item()
            )

            total_entropy += (
                entropy.item()
            )

        torch.nn.utils.clip_grad_norm_(
            self.agent.parameters(),
            max_norm=1.0
        )

        self.optimizer.step()

        num_transitions = len(
            self.memory
        )

        self.memory = []

        average_loss = (
            total_policy_loss
            + self.value_coef * total_value_loss
        ) / num_transitions

        return average_loss


def run_ppo_training(
    agent,
    chunker,
    extractor,
    hf_token,
    dataset,
    num_steps=100,
    ppo_batch_size=8
):

    if len(dataset) == 0:
        raise ValueError(
            "Training dataset is empty."
        )

    trainer = PPOTrainer(
        agent=agent,
        lr=1e-5,
        clip_param=0.2,
        gamma=0.99,
        gae_lambda=0.95,
        value_coef=0.5,
        entropy_coef=0.01,
        batch_size=ppo_batch_size
    )

    reward_history = []
    privacy_history = []
    task_gain_history = []
    loss_history = []

    reward_window = deque(
        maxlen=50
    )

    privacy_window = deque(
        maxlen=50
    )

    task_window = deque(
        maxlen=50
    )

    loss_window = deque(
        maxlen=50
    )

    print(
        f"Starting PPO training for "
        f"{num_steps} steps..."
    )

    for step in tqdm(
        range(num_steps),
        desc="PPO Training"
    ):

        sample = dataset[
            step % len(dataset)
        ]

        query = sample["query"]
        pii_entities = sample["pii_units"]
        gold_response = sample["target_response"]

        try:

            (
                embeddings,
                actions,
                log_prob,
                reward,
                state_value,
                privacy_leak,
                task_gain
            ) = execute_real_rollout(
                query,
                pii_entities,
                gold_response,
                agent,
                chunker,
                extractor,
                hf_token
            )

        except Exception as e:

            print(
                f"Rollout failed at step "
                f"{step}: {e}"
            )

            continue

        trainer.store_transition(
            embeddings,
            actions,
            log_prob,
            reward,
            state_value
        )

        reward_history.append(
            reward
        )

        privacy_history.append(
            privacy_leak
        )

        task_gain_history.append(
            task_gain
        )

        reward_window.append(
            reward
        )

        privacy_window.append(
            privacy_leak
        )

        task_window.append(
            task_gain
        )

        if (
            len(trainer.memory)
            >= trainer.batch_size
        ):

            try:

                loss = trainer.update()

                if loss is not None:

                    loss_history.append(
                        loss
                    )

                    loss_window.append(
                        loss
                    )

            except Exception as e:

                print(
                    f"PPO update failed at "
                    f"step {step}: {e}"
                )

                trainer.memory = []

        if (
            (step + 1) % 25 == 0
        ):

            print(
                "\n=============================="
            )

            print(
                f"Step {step + 1}/{num_steps}"
            )

            print(
                f"Average Reward       : "
                f"{np.mean(reward_window):.4f}"
            )

            print(
                f"Average Privacy Leak : "
                f"{100 * np.mean(privacy_window):.2f}%"
            )

            print(
                f"Average Task Gain    : "
                f"{np.mean(task_window):.4f}"
            )

            if loss_window:

                print(
                    f"Average PPO Loss     : "
                    f"{np.mean(loss_window):.4f}"
                )

            print(
                "=============================="
            )

    print(
        "\n========== FINAL RESULTS =========="
    )

    print(
        f"Total Episodes        : "
        f"{len(reward_history)}"
    )

    if reward_history:

        print(
            f"Average Reward        : "
            f"{np.mean(reward_history):.4f}"
        )

        print(
            f"Average Privacy Leak  : "
            f"{100 * np.mean(privacy_history):.2f}%"
        )

        print(
            f"Average Task Gain     : "
            f"{np.mean(task_gain_history):.4f}"
        )

    if loss_history:

        print(
            f"Average PPO Loss      : "
            f"{np.mean(loss_history):.4f}"
        )

    print(
        "==================================="
    )

    model_path = "privacy_r1_agent.pt"

    torch.save(
        agent.state_dict(),
        model_path
    )

    print(
        f"Model saved to {model_path}"
    )

    if reward_history:

        plt.figure(
            figsize=(10, 5)
        )

        plt.plot(
            reward_history
        )

        plt.title(
            "Reward During PPO Training"
        )

        plt.xlabel(
            "Episode"
        )

        plt.ylabel(
            "Reward"
        )

        plt.grid(True)

        plt.show()

    return {
        "reward_history": reward_history,
        "privacy_history": privacy_history,
        "task_gain_history": task_gain_history,
        "loss_history": loss_history
    }